<a href="https://colab.research.google.com/github/ZePequeno25/YOLO/blob/main/Treinando_YOLO_Models_PT_BR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Treine Modelos YOLO no Google Colab
**Autor:** Evan Juras, [EJ Technology Consultants](https://ejtech.io)

**Última atualização:** 3 de janeiro de 2025

**GitHub:** [Treine e Implante Modelos YOLO](https://github.com/EdjeElectronics/Train-and-Deploy-YOLO-Models)

# Introdução

Este notebook usa o [Ultralytics](https://docs.ultralytics.com/) para treinar modelos de detecção de objetos YOLO11, YOLOv8 ou YOLOv5 com um conjunto de dados personalizado. Ao final deste Colab, você terá um modelo YOLO personalizado que poderá executar em seu PC, celular ou dispositivo de borda, como o Raspberry Pi.

<p align=center>
<img src="https://s3.us-west-1.amazonaws.com/evanjuras.com/img/yolo-model-demo.gif" height="360"><br>
<i>Modelo personalizado de detecção de doces YOLO em ação!</i>
</p>

Criei um vídeo no YouTube que explica este guia passo a passo. Recomendo acompanhar o vídeo enquanto trabalha com este caderno.

<p align=center>
<a href="https://youtu.be/r0RspiLG260" target="_blank"><img src="https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/doc/Train_YOLO_Thumbnail2.png" height="240"><br>
<i>Clique aqui para assistir ao vídeo!</i></a>
</p>

**Nota importante: Este notebook será atualizado continuamente para garantir sua compatibilidade com as versões mais recentes do Ultralytics e do YOLO. Caso encontre alguma diferença entre o vídeo do YouTube e este notebook, siga sempre as instruções do notebook!**

### Trabalhando no Colab
O Colab oferece uma máquina virtual no seu navegador, completa com sistema operacional Linux, sistema de arquivos, ambiente Python e, o melhor de tudo, uma GPU gratuita. Instalaremos o PyTorch e o Ultralytics neste ambiente e o utilizaremos para treinar nosso modelo. Basta clicar no botão "Executar" nas seções de código deste notebook para executá-las na máquina virtual.

### Navegação
Para navegar neste notebook, use o sumário na barra lateral esquerda para pular de uma seção para outra.

**Verifique a disponibilidade da GPU NVIDIA**

Certifique-se de estar usando uma máquina equipada com GPU. Para isso, acesse "Runtime" -> "Change runtime type" na barra de menu superior e selecione uma das opções de GPU na seção "Hardware accelerator". Clique em "Play" no bloco de código a seguir para verificar se a GPU NVIDIA está presente e pronta para o treinamento.

In [ ]:
!nvidia-smi

#1.&nbsp;Reunir e etiquetar imagens de treinamento

Antes de iniciarmos o treinamento, precisamos coletar e rotular as imagens que serão usadas para treinar o modelo de detecção de objetos. Um bom ponto de partida para um modelo de prova de conceito é 200 imagens. As imagens de treinamento devem conter objetos aleatórios, além dos objetos desejados, e devem apresentar uma variedade de fundos e condições de iluminação.

Existem algumas opções para coletar imagens:

* Criar um conjunto de dados personalizado tirando suas próprias fotos dos objetos e rotulando-as (isso geralmente resulta no melhor desempenho)
* Encontrar um conjunto de dados pré-fabricado em fontes como [Roboflow Universe](), [Kaggle]() ou [Google Images V7]()

Se você quiser criar seu próprio conjunto de dados, existem diversas ferramentas disponíveis para rotular imagens. Uma boa opção é o [Label Studio](https://labelstud.io/?utm_source=youtube&utm_medium=video&utm_campaign=edjeelectronics), uma ferramenta de rotulagem gratuita e de código aberto que possui um fluxo de trabalho simples, ao mesmo tempo que oferece recursos mais avançados. Meu vídeo no YouTube, que explica passo a passo este caderno (link a ser adicionado em breve), mostra como etiquetar imagens com o Label Studio.

<p align=center>
<img src="https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/doc/label-studio-example.PNG" height="380"><br>
<i>Exemplo de uma imagem de doce rotulada com o Label Studio.</i>
</p>

Se você usou o Label Studio para rotular e exportar as imagens, elas serão exportadas em um arquivo `project.zip` que contém o seguinte:

- Uma pasta `images` contendo as imagens
- Uma pasta `labels` contendo os rótulos no formato de anotação YOLO
- Um arquivo `classes.txt` contendo todas as classes
- Um arquivo `notes.json` contendo informações específicas do Label Studio (este arquivo pode ser ignorado)

Se você obteve seu conjunto de dados de outra fonte (como o Roboflow Universe) ou usou outra ferramenta para rotular seu conjunto de dados, certifique-se de que os arquivos estejam organizados em a mesma estrutura de pastas.

<p align=center>
<img src="https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/doc/zipped-data-example.png" height=""><br>
<i>Organize seus dados nas pastas mostradas aqui. Veja meu <a href="https://s3.us-west-1.amazonaws.com/evanjuras.com/resources/candy_data_06JAN25.zip">Conjunto de Dados de Detecção de Doces</a> para um exemplo.</i>
</p>

Depois de criar seu conjunto de dados, colocá-lo na estrutura de arquivos mostrada acima e compactá-lo em `data.zip`, você estará pronto para prosseguir para a próxima etapa.

# 2.&nbsp;Carregar o conjunto de dados de imagens e preparar os dados de treinamento.

Em seguida, vamos carregar nosso conjunto de dados e prepará-lo para o treinamento com o YOLO. Dividiremos o conjunto de dados em pastas de treinamento e validação e geraremos automaticamente o arquivo de configuração para treinar o modelo.

## 2.1 Carregar imagens

Primeiro, precisamos carregar o conjunto de dados para o Colab. Aqui estão algumas opções para mover a pasta `data.zip` para esta instância do Colab.

**Opção 1. Carregar pelo Google Colab**

Carregue o arquivo `data.zip` para a instância do Google Colab clicando no ícone "Arquivos" no lado esquerdo do navegador e, em seguida, no ícone "Carregar para o armazenamento da sessão". Selecione a pasta zip para fazer o upload.

<p>
<br>
<img src="https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/doc/upload-colab-files.png" height="240">
</p>

**Opção 2. Copiar do Google Drive**

Você também pode enviar suas imagens para o seu Google Drive pessoal, montar a unidade nesta sessão do Colab e copiá-las para o sistema de arquivos do Colab. Essa opção é ideal se você quiser enviar as imagens com antecedência para não precisar esperar o upload ser concluído a cada reinicialização do Colab. Se você tiver mais de 50 MB de imagens, recomendo usar esta opção.

Primeiro, envie o arquivo `data.zip` para o seu Google Drive e anote a pasta onde ele foi salvo. Substitua `MeuDrive/caminho/para/data.zip` pelo caminho do seu arquivo zip. (Por exemplo, eu enviei o arquivo zip para uma pasta chamada "candy-dataset1", então usaria `MeuDrive/candy-dataset1/data.zip` como caminho). Em seguida, execute o seguinte bloco de código para montar seu Google Drive nesta sessão do Colab e copiar a pasta para este sistema de arquivos.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

!cp /content/gdrive/MyDrive/path/to/data.zip /content

**Opção 3. Use meu conjunto de dados de detecção de doces ou de moedas**

Se você quiser apenas testar o processo em um conjunto de dados pré-fabricado, pode usar um dos meus conjuntos de dados:

* [Conjunto de dados de imagens de doces](https://s3.us-west-1.amazonaws.com/evanjuras.com/resources/candy_data_14DEC24.zip), que contém 162 imagens de doces populares (Skittles, Snickers, etc.)
* [Conjunto de dados de imagens de moedas](https://s3.us-west-1.amazonaws.com/evanjuras.com/resources/YOLO_coin_data_12DEC30.zip), que contém 750 imagens de moedas americanas (centavos, moedas de dez centavos, moedas de cinco centavos e moedas de vinte e cinco centavos)

Baixe um dos conjuntos de dados executando o seguinte bloco de código. Usarei o conjunto de dados de detecção de doces como exemplo para o restante do notebook.

In [ ]:
# To use my one of pre-made dataset instead of your own custom dataset, download it here (control which dataset is downloaded by commenting out either line)
!wget -O /content/data.zip https://s3.us-west-1.amazonaws.com/evanjuras.com/resources/candy_data_06JAN25.zip # Candy dataset
#!wget -O /content/data.zip https://s3.us-west-1.amazonaws.com/evanjuras.com/resources/YOLO_coin_data_12DEC30.zip # Coin dataset

## 2.2 Divida as imagens em pastas de treino e validação.

Neste ponto, independentemente de você ter usado a Opção 1, 2 ou 3, você deverá conseguir clicar no ícone da pasta à esquerda e ver seu arquivo `data.zip` na lista de arquivos. Em seguida, vamos descompactar o arquivo `data.zip` e criar algumas pastas para armazenar as imagens. Execute o seguinte bloco de código para descompactar os dados.

In [ ]:
# Unzip images to a custom data folder
!unzip -q /content/data.zip -d /content/custom_data

O Ultralytics requer uma estrutura de pastas específica para armazenar os dados de treinamento dos modelos. A pasta raiz se chama “data”. Dentro dela, existem duas pastas principais:

* **Train**: Estas são as imagens usadas para treinar o modelo. Em cada época de treinamento, todas as imagens do conjunto de treinamento são passadas para a rede neural. O algoritmo de treinamento ajusta os pesos da rede para que se adequem aos dados das imagens.

* **Validation**: Estas imagens são usadas para verificar o desempenho do modelo ao final de cada época de treinamento.

Dentro de cada uma dessas pastas, há uma pasta “images” e uma pasta “labels”, que contêm os arquivos de imagem e os arquivos de anotação, respectivamente.

Criei um script em Python que cria automaticamente a estrutura de pastas necessária e move aleatoriamente 90% do conjunto de dados para a pasta "train" e 10% para a pasta "validation". Execute o seguinte bloco de código para baixar e executar o script.

In [ ]:
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

# TO DO: Improve robustness of train_val_split.py script so it can handle nested data folders, etc
!python train_val_split.py --datapath="/content/custom_data" --train_pct=0.9

# 3. Instalação dos Requisitos (Ultralytics)

Em seguida, instalaremos a biblioteca Ultralytics nesta instância do Google Colab. Esta biblioteca Python será usada para treinar o modelo YOLO.

In [ ]:
!pip install ultralytics

# 4.&nbsp;Configurar treinamento

Há um último passo antes de podermos executar o treinamento: precisamos criar o arquivo YAML de configuração de treinamento do Ultralytics. Este arquivo especifica a localização dos seus dados de treinamento e validação, e também define as classes do modelo. Um exemplo de arquivo de configuração está disponível [aqui](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/datasets/coco128.yaml).

Execute o bloco de código abaixo para gerar automaticamente um arquivo de configuração `data.yaml`. Certifique-se de que você tenha um arquivo labelmap localizado em `custom_data/classes.txt`. Se você usou o Label Studio ou um dos meus conjuntos de dados predefinidos, ele já deve estar presente. Se você montou o conjunto de dados de outra forma, talvez precise criar manualmente o arquivo `classes.txt` (veja [aqui](https://github.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/blob/main/doc/classes.txt) para um exemplo de como ele é formatado).

In [ ]:
# Função Python para criar automaticamente o arquivo de configuração data.yaml
# 1. Lê o arquivo "classes.txt" para obter a lista de nomes de classes
# 2. Cria um dicionário de dados com os caminhos corretos para as pastas, o número de classes e os nomes das classes
# 3. Escreve os dados em formato YAML no arquivo data.yaml

import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

 # Leia o arquivo class.txt para obter os nomes das classes
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

 # Criar dicionário de dados
  data = {
      'path': '/content/data',
      'train': 'train/images',
      'val': 'validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Escrever dados em arquivo YAML
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Defina o caminho para classes.txt e execute a função
path_to_classes_txt = '/content/custom_data/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml

# 5.&nbsp;Treinando Modelo

## 5.1 Parâmetros de Treinamento
Agora que os dados estão organizados e o arquivo de configuração foi criado, estamos prontos para começar o treinamento! Primeiro, há alguns parâmetros importantes a serem definidos. Visite meu artigo sobre [Treinamento de Modelos YOLO Localmente](https://www.ejtech.io/learn/train-yolo-models) para saber mais sobre esses parâmetros e como escolhê-los.

**Arquitetura e tamanho do modelo (`model`):**

Existem vários tamanhos de modelos YOLO11 disponíveis para treinamento, incluindo `yolo11n.pt`, `yolo11s.pt`, `yolo11m.pt`, `yolo11l.pt` e `yolo11xl.pt`. Modelos maiores são executados mais lentamente, mas têm maior precisão, enquanto modelos menores são executados mais rapidamente, mas têm menor precisão. Fiz um breve vídeo no YouTube que compara o desempenho de diferentes modelos YOLO em um Raspberry Pi 5 e em um laptop com uma GPU RTX 4050 [confira aqui para ter uma ideia da velocidade e precisão deles](https://youtu.be/_WKS4E9SmkA). Se você não tiver certeza sobre qual tamanho de modelo usar, `yolo11s.pt` é um bom ponto de partida.

Você também pode treinar modelos YOLOv8 ou YOLOv5 substituindo `yolo11` por `yolov8` ou `yolov5`.

**Número de épocas (`epochs`)**

Em aprendizado de máquina, uma “época” é uma única passagem por todo o conjunto de dados de treinamento. Definir o número de épocas determina por quanto tempo o modelo será treinado. A quantidade ideal de épocas depende do tamanho do conjunto de dados e da arquitetura do modelo. Se o seu conjunto de dados tiver menos de 200 imagens, um bom ponto de partida é 60 épocas. Se o seu conjunto de dados tiver mais de 200 imagens, um bom ponto de partida é 40 épocas.

**Resolução (`imgsz`)**

A resolução tem um grande impacto na velocidade e na precisão do modelo: um modelo com resolução mais baixa terá maior velocidade, mas menor precisão. Os modelos YOLO são normalmente treinados e inferidos em uma resolução de 640x640. No entanto, se você quiser que seu modelo seja executado mais rapidamente ou souber que trabalhará com imagens de baixa resolução, tente usar uma resolução menor, como 480x480.

## 5.2 Iniciando Treinamento

Execute o seguinte bloco de código para iniciar o treinamento. Se desejar usar um modelo diferente, um número diferente de épocas ou uma resolução diferente, altere `model`, `epochs` ou `imgsz`.

In [ ]:
!yolo detect train data=/content/data.yaml model=yolo11s.pt epochs=60 imgsz=640

O algoritmo de treinamento analisará as imagens nos diretórios de treinamento e validação e, em seguida, iniciará o treinamento do modelo. Ao final de cada época de treinamento, o programa executa o modelo no conjunto de dados de validação e reporta o mAP, a precisão e a revocação resultantes. Conforme o treinamento prossegue, o mAP geralmente aumenta a cada época. O treinamento será encerrado após atingir o número de épocas especificado por `epochs`.

> **NOTA:** Certifique-se de permitir que o treinamento seja concluído, pois um otimizador é executado ao final do treinamento para remover camadas desnecessárias do modelo.

Os melhores pesos do modelo treinado serão salvos em `content/runs/detect/train/weights/best.pt`. Informações adicionais sobre o treinamento são salvas na pasta `content/runs/detect/train`, incluindo um arquivo `results.png` que mostra a evolução da perda, da precisão, da revocação e do mAP em cada época.

#6.&nbsp;Testando o Modelo

O modelo foi treinado; agora é hora de testá-lo! Os comandos abaixo executam o modelo nas imagens da pasta de validação e exibem os resultados das 10 primeiras imagens. Esta é uma boa maneira de confirmar se o seu modelo está funcionando conforme o esperado. Clique em "Reproduzir" nos blocos abaixo para ver o desempenho do seu modelo.

In [ ]:
!yolo detect predict model=runs/detect/train/weights/best.pt source=data/validation/images save=True

In [ ]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(f'/content/runs/detect/predict/*.jpg')[:10]:
  display(Image(filename=image_path, height=400))
  print('\n')


O modelo deve desenhar uma caixa ao redor de cada objeto de interesse em cada imagem. Se ele não estiver detectando objetos corretamente, aqui estão algumas dicas:

1. Verifique novamente seu conjunto de dados para garantir que não haja erros de rotulagem ou exemplos conflitantes.

2. Aumente o número de épocas usadas para o treinamento.

3. Use um modelo de tamanho maior (por exemplo, `yolo11l.pt`).

4. Adicione mais imagens ao conjunto de dados de treinamento. Veja meu [vídeo sobre o conjunto de dados](https://www.youtube.com/watch?v=v0ssiOY6cfg) para dicas sobre como capturar boas imagens de treinamento e melhorar a precisão.

Você também pode executar o modelo em arquivos de vídeo ou outras imagens, enviando-os para este notebook e usando o comando `!yolo detect predict` acima, onde `source` aponta para o local do arquivo de vídeo, imagem ou pasta de imagens. Os resultados serão salvos em `runs/detect/predict`.

Desenhar caixas em imagens é ótimo, mas não é muito útil por si só. Também não é muito útil executar esses modelos apenas dentro de um notebook do Colab: é mais fácil executá-los em um computador local. Continue para a próxima seção para ver como baixar seu modelo recém-treinado e executá-lo em um dispositivo local.

#7.&nbsp;Implementando o Modelo

Agora que seu modelo personalizado foi treinado, ele está pronto para ser baixado e implantado em um aplicativo! Os modelos YOLO podem ser executados em uma ampla variedade de hardware, incluindo PCs, sistemas embarcados e celulares. O Ultralytics facilita a conversão dos modelos YOLO para vários formatos (`tflite`, `onnx`, etc.) e sua implantação em diversos ambientes.

Esta seção mostra como baixar o modelo e fornece links para instruções de implantação em seu PC e dispositivos de borda, como o Raspberry Pi.

## 7.1 Baixar o Modelo YOLO

* Item da lista
* Item da lista

Primeiro, compacte e baixe o modelo treinado executando os blocos de código abaixo.

O código cria uma pasta chamada `my_model`, move os pesos do modelo para dentro dela e os renomeia de `best.pt` para `my_model.pt`. Ele também adiciona os resultados do treinamento, caso você queira consultá-los posteriormente. Em seguida, compacta a pasta como `my_model.zip`.

In [ ]:
# Crie a pasta "my_model" para armazenar os pesos do modelo e os resultados do treinamento.
!mkdir /content/my_model
!cp /content/runs/detect/train/weights/best.pt /content/my_model/my_model.pt
!cp -r /content/runs/detect/train /content/my_model

# Compacte em "my_model.zip"
%cd my_model
!zip /content/my_model.zip my_model.pt
!zip -r /content/my_model.zip train
%cd /content

In [ ]:
# Por algum motivo, isso demora uma eternidade. Você também pode baixar o modelo da barra lateral.
from google.colab import files

files.download('/content/my_model.zip')

## 7.2 Implantação do Modelo YOLO em Dispositivos Locais

A seguir, pegaremos o modelo baixado e o executaremos em um dispositivo local. Esta seção fornece instruções sobre como implantar modelos YOLO em diversos dispositivos.

Criei um script básico em Python, `yolo_detect.py`, que demonstra como carregar um modelo, executar inferência em uma imagem de origem, analisar os resultados da inferência e exibir caixas ao redor de cada classe detectada na imagem. O script fornece um exemplo de como trabalhar com modelos YOLO da Ultralytics em Python e pode ser usado como ponto de partida para aplicações mais avançadas.

### 7.2.1 Implantação em PC (Windows, Linux ou macOS)

A maneira mais fácil de executar modelos Ultralytics em um PC é usando o Anaconda. O Anaconda configura um ambiente virtual Python e permite que você instale facilmente o Ultralytics e o PyTorch. Ele instala automaticamente o CUDA e o cuDNN, o que permite acelerar a inferência do modelo com sua GPU NVIDIA.

> **NOTA:** Meu vídeo no YouTube (link a ser adicionado) mostra como implantar seu modelo em um PC. Ele descreve os seguintes passos, então assista ao vídeo se preferir instruções visuais.

**1. Baixe e instale o Anaconda**

Acesse a página de download do Anaconda em https://anaconda.com/download, clique no botão “pular registro” e baixe o pacote para o seu sistema operacional. Quando o download terminar, execute o instalador e siga as etapas de instalação. Você pode usar as opções padrão para a instalação.

**2. Configurar ambiente virtual**

Após a instalação, execute o Anaconda Prompt na barra Iniciar. (Se estiver usando macOS ou Linux, basta abrir um terminal de comando).

Execute os seguintes comandos para criar e ativar um novo ambiente Python:

```
conda create --name yolo-env1 python=3.12 -y
conda activate yolo-env1
```

Instale o Ultralytics (que também instala bibliotecas de importação como OpenCV-Python, Numpy e PyTorch) executando o seguinte comando:

```
pip install ultralytics
```

Se você tiver uma GPU NVIDIA, poderá instalar a versão do PyTorch compatível com GPU executando o seguinte comando:

```
pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
```

**3. Extraia o modelo baixado**
Pegue o arquivo `my_model.zip` que você baixou na Etapa 7.1 e descompacte-o em uma pasta no seu computador. No terminal do Anaconda Prompt, acesse a pasta descompactada usando:

```
cd caminho/para/a/pasta
```

**4. Baixe e execute o yolo_detect.py**

Baixe o script `yolo_detect.py` para a pasta `my_model` usando:

```
curl -o yolo_detect.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/yolo_detect.py
```

Pronto! Agora podemos executar o script. Para executar a inferência com um modelo YOLOv8s em uma câmera USB com resolução de 1280x720, execute o seguinte comando:

```
python yolo_detect.py --model my_model.pt --source usb0 --resolution 1280x720
```

Uma janela será exibida mostrando a transmissão ao vivo da sua webcam com caixas desenhadas ao redor dos objetos detectados em cada quadro.

Você também pode executar o modelo em um arquivo de vídeo, imagem ou pasta de imagens. Para ver a lista completa de argumentos para `yolo_detect.py`, execute `python yolo_detect.py --help` ou consulte o arquivo [README](https://github.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/blob/main/README.md).

### 7.2.2 Implantação no Raspberry Pi

Fique de olho em um artigo que mostrará como converter modelos YOLO para o formato NCNN e executá-los no Raspberry Pi!

# 8.&nbsp;Conclusão

Parabéns! Você treinou e implantou com sucesso um modelo de detecção de objetos YOLO. 😀

Em seguida, você pode expandir sua aplicação além de simplesmente desenhar caixas e contar objetos. Adicione funcionalidades como registrar o número de objetos detectados ao longo do tempo ou tirar uma foto quando determinados objetos forem detectados. Confira alguns exemplos de aplicações em nosso repositório do GitHub: https://github.com/EdjeElectronics/Train-and-Deploy-YOLO-Models

Obrigado por trabalhar com este notebook e boa sorte com seus projetos!

# Apêndice: Erros comuns

Se você encontrar algum erro ao seguir este notebook, faça o seguinte:

- Verifique se os arquivos do conjunto de dados estão configurados na estrutura de pastas correta.
- Certifique-se de que não haja erros de digitação ou outros erros no seu arquivo labelmap.
- Pesquise o erro no Google para encontrar soluções.

Se nenhuma dessas opções funcionar, abra uma [Issue](https://github.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/issues) na página do GitHub. Nesta seção, adicionarei soluções para erros comuns à medida que forem surgindo.